# Semi-Supervised Model Comparison

This notebook compares supervised classifiers trained on **Isolation Forest-derived pseudo-labels** against the IF baseline.

**Workflow:**
1. Load the trained IF model and generate angriness labels (1–5) for each split
2. Train **Random Forest** and **XGBoost** classifiers on those labels
3. Compare all models on proxy R², Spearman correlations, accuracy, and F1
4. Test with both 7 behavioral features and all 22 features

**Why semi-supervised?** The IF model only explains ~24% of tilt variance (proxy R²=0.24). Supervised models can learn complex feature interactions that IF misses, potentially producing cleaner tilt predictions.

In [1]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from xgboost import XGBClassifier

TRAINER_DIR = Path.cwd().resolve().parent
PROCESSED_DIR = TRAINER_DIR / "data" / "processed"
MODELS_DIR = TRAINER_DIR / "models"
PLOT_DIR = Path.cwd() / "plots"
PLOT_DIR.mkdir(exist_ok=True)

print(f"Trainer root: {TRAINER_DIR}")

Trainer root: C:\swe_repos\my-portfolio-and-vps-repos\Chess-Assistance\machine_learning\trainers\trainer-angriness-predictor


## Load Data and IF Model

In [2]:
df_train = pd.read_csv(PROCESSED_DIR / "features_train.csv")
df_val = pd.read_csv(PROCESSED_DIR / "features_val.csv")
df_test = pd.read_csv(PROCESSED_DIR / "features_test.csv")
raw_train = pd.read_csv(PROCESSED_DIR / "raw_train.csv")
raw_val = pd.read_csv(PROCESSED_DIR / "raw_val.csv")
raw_test = pd.read_csv(PROCESSED_DIR / "raw_test.csv")

if_model = joblib.load(MODELS_DIR / "model.pkl")
with open(MODELS_DIR / "angriness_bins.json") as f:
    bins_data = json.load(f)
bin_edges = bins_data["bin_edges"]
MODEL_FEATURES = bins_data["model_features"]

print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")
print(f"IF model features ({len(MODEL_FEATURES)}): {MODEL_FEATURES}")
print(f"All features ({len(df_train.columns)}): {list(df_train.columns)}")

Train: (16159, 22), Val: (4040, 22), Test: (5050, 22)
IF model features (22): ['consecutive_losses_pregame', 'avg_tpm_seconds_player', 'blunder_cnt_player', 'mistake_cnt_player', 'inaccuracy_cnt_player', 'acpl_player', 'accuracy_player', 'elo', 'elo_diff', 'opponent_elo', 'elo_gap', 'time_control_initial', 'time_control_increment', 'move_cnt', 'move_cnt_player', 'sleep_duration', 'awaken_duration', 'avg_ppm', 'avg_celsius', 'water_intake_ml', 'avg_lux', 'is_black']
All features (22): ['consecutive_losses_pregame', 'avg_tpm_seconds_player', 'blunder_cnt_player', 'mistake_cnt_player', 'inaccuracy_cnt_player', 'acpl_player', 'accuracy_player', 'elo', 'elo_diff', 'opponent_elo', 'elo_gap', 'time_control_initial', 'time_control_increment', 'move_cnt', 'move_cnt_player', 'sleep_duration', 'awaken_duration', 'avg_ppm', 'avg_celsius', 'water_intake_ml', 'avg_lux', 'is_black']


c:\Users\konkd\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\konkd\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Generate IF Pseudo-Labels

In [3]:
def score_to_angriness(score, bin_edges):
    for i in range(len(bin_edges) - 1):
        if score <= bin_edges[i + 1]:
            return 5 - i
    return 1


def generate_labels(model, features_df, model_features, bin_edges):
    scores = model.decision_function(features_df[model_features].values)
    return np.array([score_to_angriness(s, bin_edges) for s in scores])


y_train = generate_labels(if_model, df_train, MODEL_FEATURES, bin_edges)
y_val = generate_labels(if_model, df_val, MODEL_FEATURES, bin_edges)
y_test = generate_labels(if_model, df_test, MODEL_FEATURES, bin_edges)

print("Label distribution:")
for name, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
    counts = pd.Series(y).value_counts().sort_index()
    print(f"  {name}: {dict(counts)}")

AttributeError: 'RandomForestClassifier' object has no attribute 'decision_function'

## Evaluation Helpers

In [ ]:
COMPOSITE_COLS = ["acpl_player", "blunder_cnt_player", "consecutive_losses_pregame"]


def compute_composite_tilt(raw_df):
    components = []
    for c in COMPOSITE_COLS:
        vals = raw_df[c].values.astype(float)
        vmin, vmax = vals.min(), vals.max()
        components.append((vals - vmin) / (vmax - vmin + 1e-9))
    return np.mean(components, axis=0)


def evaluate_model(name, y_true, y_pred, raw_df):
    composite = compute_composite_tilt(raw_df)
    r, _ = pearsonr(y_pred.astype(float), composite)
    proxy_r2 = round(float(r ** 2), 4)

    spearman = {}
    for col in COMPOSITE_COLS:
        corr, _ = spearmanr(y_pred.astype(float), raw_df[col].values)
        spearman[col] = round(float(corr), 4)

    acc = round(accuracy_score(y_true, y_pred), 4)
    f1 = round(f1_score(y_true, y_pred, average="weighted"), 4)

    return {
        "name": name,
        "accuracy": acc,
        "f1_weighted": f1,
        "proxy_r2": proxy_r2,
        "spearman_acpl": spearman["acpl_player"],
        "spearman_blunders": spearman["blunder_cnt_player"],
        "spearman_consloss": spearman["consecutive_losses_pregame"],
    }


print("Evaluation helpers ready.")

Evaluation helpers ready.


## IF Baseline

In [ ]:
results = []

# IF baseline: labels ARE the predictions (accuracy=1.0 by definition on train)
# For fair comparison, evaluate IF on test using proxy metrics
if_test_result = evaluate_model("IF baseline (7feat)", y_test, y_test, raw_test)
if_test_result["accuracy"] = "-"
if_test_result["f1_weighted"] = "-"
results.append(if_test_result)

print(f"IF baseline proxy R2: {if_test_result['proxy_r2']}")
print(f"IF baseline Spearman: ACPL={if_test_result['spearman_acpl']}, "
      f"Blunders={if_test_result['spearman_blunders']}, ConsLoss={if_test_result['spearman_consloss']}")

IF baseline proxy R2: nan
IF baseline Spearman: ACPL=nan, Blunders=nan, ConsLoss=nan


C:\Users\konkd\AppData\Local\Temp\ipykernel_15556\3849760312.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_pred.astype(float), composite)
C:\Users\konkd\AppData\Local\Temp\ipykernel_15556\3849760312.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(y_pred.astype(float), raw_df[col].values)


## Random Forest Classifier

In [ ]:
for feat_name, feat_cols in [("7feat", MODEL_FEATURES), ("22feat", list(df_train.columns))]:
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(df_train[feat_cols].values, y_train)

    y_pred_val = rf.predict(df_val[feat_cols].values)
    y_pred_test = rf.predict(df_test[feat_cols].values)

    val_result = evaluate_model(f"RF {feat_name} (val)", y_val, y_pred_val, raw_val)
    test_result = evaluate_model(f"RF {feat_name}", y_test, y_pred_test, raw_test)
    results.append(test_result)

    print(f"\n=== Random Forest ({feat_name}) ===")
    print(f"  Val  accuracy={val_result['accuracy']}, F1={val_result['f1_weighted']}, "
          f"proxy_R2={val_result['proxy_r2']}, Spearman(ConsLoss)={val_result['spearman_consloss']}")
    print(f"  Test accuracy={test_result['accuracy']}, F1={test_result['f1_weighted']}, "
          f"proxy_R2={test_result['proxy_r2']}, Spearman(ConsLoss)={test_result['spearman_consloss']}")

    if feat_name == "7feat":
        print(f"\n  Classification report (test):")
        print(classification_report(y_test, y_pred_test, target_names=[f"Level {i}" for i in range(1, 6)]))


=== Random Forest (7feat) ===
  Val  accuracy=1.0, F1=1.0, proxy_R2=nan, Spearman(ConsLoss)=nan
  Test accuracy=1.0, F1=1.0, proxy_R2=nan, Spearman(ConsLoss)=nan

  Classification report (test):


C:\Users\konkd\AppData\Local\Temp\ipykernel_15556\3849760312.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_pred.astype(float), composite)
C:\Users\konkd\AppData\Local\Temp\ipykernel_15556\3849760312.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(y_pred.astype(float), raw_df[col].values)
C:\Users\konkd\AppData\Local\Temp\ipykernel_15556\3849760312.py:15: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = pearsonr(y_pred.astype(float), composite)
C:\Users\konkd\AppData\Local\Temp\ipykernel_15556\3849760312.py:20: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = spearmanr(y_pred.astype(float), raw_df[col].values)


ValueError: Number of classes, 1, does not match size of target_names, 5. Try specifying the labels parameter

## XGBoost Classifier

In [ ]:
for feat_name, feat_cols in [("7feat", MODEL_FEATURES), ("22feat", list(df_train.columns))]:
    xgb = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss",
        verbosity=0,
    )
    # XGBoost requires 0-indexed classes
    xgb.fit(df_train[feat_cols].values, y_train - 1)

    y_pred_val = xgb.predict(df_val[feat_cols].values) + 1
    y_pred_test = xgb.predict(df_test[feat_cols].values) + 1

    val_result = evaluate_model(f"XGB {feat_name} (val)", y_val, y_pred_val, raw_val)
    test_result = evaluate_model(f"XGB {feat_name}", y_test, y_pred_test, raw_test)
    results.append(test_result)

    print(f"\n=== XGBoost ({feat_name}) ===")
    print(f"  Val  accuracy={val_result['accuracy']}, F1={val_result['f1_weighted']}, "
          f"proxy_R2={val_result['proxy_r2']}, Spearman(ConsLoss)={val_result['spearman_consloss']}")
    print(f"  Test accuracy={test_result['accuracy']}, F1={test_result['f1_weighted']}, "
          f"proxy_R2={test_result['proxy_r2']}, Spearman(ConsLoss)={test_result['spearman_consloss']}")

    if feat_name == "7feat":
        print(f"\n  Classification report (test):")
        print(classification_report(y_test, y_pred_test, target_names=[f"Level {i}" for i in range(1, 6)]))

## Comparison Table

In [ ]:
print("=== Test Set Comparison (all models) ===\n")
header = f"{'Model':<22} {'Acc':>6} {'F1':>6} {'R2':>6} {'ACPL':>7} {'Blund':>7} {'CLoss':>7}"
print(header)
print("-" * len(header))
for r in results:
    acc = f"{r['accuracy']}" if isinstance(r['accuracy'], str) else f"{r['accuracy']:.4f}"
    f1 = f"{r['f1_weighted']}" if isinstance(r['f1_weighted'], str) else f"{r['f1_weighted']:.4f}"
    print(f"{r['name']:<22} {acc:>6} {f1:>6} {r['proxy_r2']:>6.4f} "
          f"{r['spearman_acpl']:>7.4f} {r['spearman_blunders']:>7.4f} {r['spearman_consloss']:>7.4f}")

best = max([r for r in results if r['accuracy'] != '-'], key=lambda r: r['proxy_r2'])
print(f"\nBest model by proxy R2: {best['name']} (R2={best['proxy_r2']})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = [r["name"] for r in results]
colors = ["#95a5a6", "#3498db", "#2980b9", "#e74c3c", "#c0392b"]

# Proxy R2
r2_vals = [r["proxy_r2"] for r in results]
axes[0].barh(model_names, r2_vals, color=colors, edgecolor="white")
axes[0].set_xlabel("Proxy R\u00b2")
axes[0].set_title("Proxy R\u00b2 (higher = better)")
for i, v in enumerate(r2_vals):
    axes[0].text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=9)

# Spearman ConsLoss
cl_vals = [r["spearman_consloss"] for r in results]
axes[1].barh(model_names, cl_vals, color=colors, edgecolor="white")
axes[1].set_xlabel("Spearman \u03c1")
axes[1].set_title("Spearman ConsLoss (higher = better)")
for i, v in enumerate(cl_vals):
    axes[1].text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=9)

# Spearman Blunders
bl_vals = [r["spearman_blunders"] for r in results]
axes[2].barh(model_names, bl_vals, color=colors, edgecolor="white")
axes[2].set_xlabel("Spearman \u03c1")
axes[2].set_title("Spearman Blunders (higher = better)")
for i, v in enumerate(bl_vals):
    axes[2].text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=9)

fig.suptitle("Semi-Supervised Model Comparison (Test Set)", fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOT_DIR / "supervised_comparison.png", dpi=150)
plt.show()

## Sanity Check — Best Model Per-Level Stats

In [ ]:
# Retrain best model to get its predictions for sanity check
best_name = best["name"]
print(f"Sanity check for: {best_name}\n")

is_xgb = "XGB" in best_name
is_22 = "22feat" in best_name
feat_cols = list(df_train.columns) if is_22 else MODEL_FEATURES

if is_xgb:
    best_model = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, eval_metric="mlogloss", verbosity=0)
    best_model.fit(df_train[feat_cols].values, y_train - 1)
    best_pred_test = best_model.predict(df_test[feat_cols].values) + 1
else:
    best_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    best_model.fit(df_train[feat_cols].values, y_train)
    best_pred_test = best_model.predict(df_test[feat_cols].values)

tilt_cols = ["acpl_player", "blunder_cnt_player", "consecutive_losses_pregame"]
raw = raw_test.copy()
raw["_pred"] = best_pred_test

sanity = raw.groupby("_pred")[tilt_cols].mean().round(2)
sanity.index.name = "Predicted Level"
sanity.columns = ["ACPL", "Blunders", "Consecutive Losses"]
print("Per-level feature means (test set):\n")
print(sanity.to_string())

print(f"\nTilt trend preserved: ACPL L5>L1={'PASS' if sanity['ACPL'].iloc[-1] > sanity['ACPL'].iloc[0] else 'FAIL'}, "
      f"Blunders={'PASS' if sanity['Blunders'].iloc[-1] > sanity['Blunders'].iloc[0] else 'FAIL'}, "
      f"ConsLoss={'PASS' if sanity['Consecutive Losses'].iloc[-1] > sanity['Consecutive Losses'].iloc[0] else 'FAIL'}")

## Summary

In [ ]:
print("=== Semi-Supervised Comparison Summary ===\n")
print(f"Best model: {best['name']}")
print(f"  Proxy R2:          {best['proxy_r2']}")
print(f"  Spearman ACPL:     {best['spearman_acpl']}")
print(f"  Spearman Blunders: {best['spearman_blunders']}")
print(f"  Spearman ConsLoss: {best['spearman_consloss']}")
if best['accuracy'] != '-':
    print(f"  Accuracy:          {best['accuracy']}")
    print(f"  F1 (weighted):     {best['f1_weighted']}")

if_r2 = results[0]['proxy_r2']
improvement = best['proxy_r2'] - if_r2
print(f"\nImprovement over IF baseline: R2 {if_r2} -> {best['proxy_r2']} ({improvement:+.4f})")
print(f"\nRecommendation: {'Update pipeline to use ' + best['name'] if improvement > 0.01 else 'Keep IF baseline'}")